In [2]:
import sys
import numpy as np
from scipy.spatial import cKDTree

sys.path.append("..")
import os
import s3fs
import pandas as pd
import geopandas as gpd

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]},
    skip_instance_cache=True,
)

dossier = "acharki-ensae/Projet-analyse-BPE/processed"
with fs.open(f"{dossier}/bpe_alim_2025.parquet", "rb") as f:
    bpe = gpd.read_parquet(f)
with fs.open(f"{dossier}/dvf_ventes_2025.parquet", "rb") as f:
    dvf = gpd.read_parquet(f)

print("BPE :", bpe.shape, bpe.crs)
print("DVF :", dvf.shape, dvf.crs)

PermissionError: Forbidden

In [2]:
!pip install scipy


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
codes_b2 = sorted(c for c in bpe["TYPEQU"].unique() if c.startswith("B2"))

types_commerces = {
    "hyper": ["B104"],
    "super": ["B105"],
    "proximite": ["B201", "B202"],
    "specialise": [c for c in codes_b2 if c not in ["B201", "B202"]],
}

for nom, codes in types_commerces.items():
    print(nom, ":", codes, "->", bpe["TYPEQU"].isin(codes).sum(), "commerces")

hyper : ['B104'] -> 2081 commerces
super : ['B105'] -> 12106 commerces
proximite : ['B201', 'B202'] -> 30577 commerces
specialise : ['B204', 'B205', 'B206', 'B207', 'B208', 'B209', 'B210'] -> 90822 commerces


In [4]:
for nom, codes in types_commerces.items():
    commerces = bpe.loc[bpe["TYPEQU"].isin(codes), ["geometry"]]

    plus_proche_type = gpd.sjoin_nearest(
        dvf[["geometry"]], commerces,
        how="left", distance_col=f"dist_{nom}",
    )
    plus_proche_type = plus_proche_type[~plus_proche_type.index.duplicated()]

    dvf[f"dist_{nom}"] = plus_proche_type[f"dist_{nom}"]
    print(f"{nom} : distance médiane {dvf[f'dist_{nom}'].median():.0f} m")

hyper : distance médiane 2309 m
super : distance médiane 749 m
proximite : distance médiane 475 m
specialise : distance médiane 284 m


In [5]:
commerces_alim = bpe.loc[
    bpe["TYPEQU"].isin(sum(types_commerces.values(), [])),
    ["TYPEQU", "geometry"],
]

plus_proche = gpd.sjoin_nearest(
    dvf[["geometry"]], commerces_alim,
    how="left", distance_col="dist_plus_proche",
)
plus_proche = plus_proche[~plus_proche.index.duplicated()]

noms_types = {code: nom for nom, codes in types_commerces.items() for code in codes}
dvf["dist_plus_proche"] = plus_proche["dist_plus_proche"]
dvf["type_plus_proche"] = plus_proche["TYPEQU"].map(noms_types)

dvf["type_plus_proche"].value_counts(normalize=True).round(3)

type_plus_proche
specialise    0.650
proximite     0.239
super         0.099
hyper         0.012
Name: proportion, dtype: float64

In [6]:
colonnes_dist = ["dist_hyper", "dist_super", "dist_proximite", "dist_plus_proche"]
dvf[colonnes_dist].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

,dist_hyper,dist_super,dist_proximite,dist_plus_proche
count,929786.0,929786.0,929786.0,929786.0
mean,5080.0,1940.0,1320.0,627.0
std,7408.0,5003.0,4652.0,4356.0
min,0.0,0.0,0.0,0.0
10%,646.0,168.0,81.0,43.0
50%,2309.0,749.0,475.0,237.0
90%,13516.0,5770.0,4015.0,1816.0
max,4082630.0,4076927.0,4075887.0,4075887.0


In [1]:
densite7_communes = bpe.groupby("DEPCOM")["DENS7"].first()
dvf["DENS7"] = dvf["code_commune"].map(densite7_communes)

print(dvf["DENS7"].value_counts(dropna=False).sort_index())

NameError: name 'bpe' is not defined

In [8]:
dvf["densite_imputee"] = dvf["DENS7"].isna()
print("Ventes sans densité connue :", dvf["densite_imputee"].sum())

dvf["DENS7"] = dvf["DENS7"].fillna("6")
dvf["DENS7"].value_counts().sort_index()

Ventes sans densité connue : 75502


DENS7
1    340021
2    136189
3     62953
4     75593
5    136356
6    166274
7     12400
Name: count, dtype: int64

In [9]:
seuil_suspect = dvf.groupby("DENS7")["dist_plus_proche"].transform(lambda x: x.quantile(0.999))
suspects = dvf[dvf["dist_plus_proche"] > seuil_suspect]

print("Ventes suspectes :", len(suspects))
suspects.sort_values("dist_plus_proche", ascending=False)[
    ["nom_commune", "code_commune", "DENS7", "dist_plus_proche", "longitude", "latitude"]
].head(20)

Ventes suspectes : 934


,nom_commune,code_commune,DENS7,dist_plus_proche,longitude,latitude
3531804,Issy-les-Moulineaux,92040,1,4.075887e+06,0.26946330000001345,83.51765750293683
3533960,Vaucresson,92076,1,1.694480e+05,2.287486500000001,40.88208410495443
3529719,Vaucresson,92076,1,1.692789e+05,2.2850567,40.883919320638924
1562027,Bourriot-Bergonce,40053,6,1.443265e+04,-0.18695199999999845,44.12140137672371
1577169,Bourriot-Bergonce,40053,6,1.436851e+04,-0.191450299999999,44.124067278938895
1557678,Bourriot-Bergonce,40053,6,1.434962e+04,-0.16089099999999895,44.11688231497875
1555598,Bourriot-Bergonce,40053,6,1.425160e+04,-0.19820114999999872,44.152456314334046
1973506,Saint-Ouen-Domprot,51508,6,1.342765e+04,4.4125368,48.60957492554793
1974353,Saint-Ouen-Domprot,51508,6,1.325279e+04,4.40783025,48.61017544988472
2445951,Borce,64136,6,1.289426e+04,-0.5578281999999994,42.857609094921365


In [10]:
categories = list(types_commerces)

def calculer_rayons(q):
    """Rayon par niveau de densité et par catégorie : quantile q de la distance au plus proche."""
    return pd.DataFrame({
        cat: dvf.groupby("DENS7")[f"dist_{cat}"].quantile(q) for cat in categories
    })

rayons = calculer_rayons(0.99)
rayons.round(0)

,hyper,super,proximite,specialise
DENS7,,,,
1,4584.0,2014.0,1607.0,928.0
2,14425.0,3458.0,2928.0,1723.0
3,23320.0,6255.0,5682.0,2833.0
4,12251.0,5400.0,4597.0,2386.0
5,30776.0,11237.0,9603.0,4027.0
6,33593.0,15529.0,11210.0,7503.0
7,37757.0,18612.0,11208.0,8103.0


In [11]:
TAILLE_BLOC = 5_000

def indice_lineaire(xy_ventes, rayon_vente, arbre, rayons_commerces):
    """Somme sur les commerces de max(0, 1 - d/R), avec R = min(rayon du logement, rayon du commerce)."""
    resultat = np.zeros(len(xy_ventes))
    for debut in range(0, len(xy_ventes), TAILLE_BLOC):
        bloc = xy_ventes[debut:debut + TAILLE_BLOC]
        paires = cKDTree(bloc).sparse_distance_matrix(
            arbre, max_distance=rayon_vente, output_type="ndarray"
        )
        rayon_paire = np.minimum(rayon_vente, rayons_commerces[paires["j"]])
        contributions = np.clip(1 - paires["v"] / rayon_paire, 0, None)
        resultat[debut:debut + len(bloc)] = np.bincount(
            paires["i"], weights=contributions, minlength=len(bloc)
        )
    return resultat


def calculer_indices(df, rayons_q, suffixe=""):
    """Calcule l'indice de chaque catégorie pour les ventes de df, avec les rayons donnés."""
    resultats = pd.DataFrame(index=df.index)
    for cat in categories:
        commerces = bpe[bpe["TYPEQU"].isin(types_commerces[cat])]
        arbre = cKDTree(np.column_stack([commerces.geometry.x, commerces.geometry.y]))
        rayons_commerces = commerces["DENS7"].map(rayons_q[cat]).to_numpy(dtype=float)

        colonne = f"acces_{cat}{suffixe}"
        resultats[colonne] = np.nan
        for niveau, rayon in rayons_q[cat].items():
            masque = (df["DENS7"] == niveau).to_numpy()
            if masque.sum() == 0:
                continue
            xy = np.column_stack([df.geometry.x[masque], df.geometry.y[masque]])
            resultats.loc[masque, colonne] = indice_lineaire(xy, rayon, arbre, rayons_commerces)
        print(f"{colonne} : terminé")
    return resultats


print("Commerces sans densité :", bpe["DENS7"].isna().sum())

Commerces sans densité : 0


In [12]:
indices = calculer_indices(dvf, rayons)
dvf[indices.columns] = indices

acces_hyper : terminé
acces_super : terminé
acces_proximite : terminé
acces_specialise : terminé


In [13]:
colonnes_acces = [f"acces_{cat}" for cat in categories]

print("Valeurs manquantes :")
print(dvf[colonnes_acces].isna().sum(), "\n")

print("Part de logements sans aucun commerce de la catégorie dans le rayon :")
(dvf[colonnes_acces] == 0).groupby(dvf["DENS7"]).mean().round(3)

Valeurs manquantes :
acces_hyper         0
acces_super         0
acces_proximite     0
acces_specialise    0
dtype: int64 

Part de logements sans aucun commerce de la catégorie dans le rayon :


,acces_hyper,acces_super,acces_proximite,acces_specialise
DENS7,,,,
1,0.010,0.010,0.010,0.010
2,0.013,0.011,0.013,0.010
3,0.036,0.013,0.012,0.011
4,0.036,0.028,0.031,0.012
5,0.047,0.026,0.016,0.011
6,0.063,0.058,0.021,0.030
7,0.145,0.109,0.012,0.020


In [14]:
dvf.groupby("DENS7")[colonnes_acces].median().round(2)

,acces_hyper,acces_super,acces_proximite,acces_specialise
DENS7,,,,
1,2.31,4.43,9.93,7.86
2,2.34,3.18,6.42,11.84
3,1.78,2.43,3.47,8.76
4,1.58,1.78,2.18,4.02
5,1.51,1.68,2.50,3.84
6,1.20,1.18,1.91,2.16
7,0.73,0.75,1.64,1.87


In [15]:
for cat in categories:
    dvf[f"rang_{cat}"] = dvf.groupby("DENS7")[f"acces_{cat}"].rank(pct=True)

In [16]:
echantillon = dvf.sample(100_000, random_state=0)

sensibilite = calculer_indices(echantillon, calculer_rayons(0.95), "_p95").join(
    calculer_indices(echantillon, calculer_rayons(0.90), "_p90")
)

pd.DataFrame([
    {
        "categorie": cat,
        "p99 vs p95": echantillon[f"acces_{cat}"].corr(sensibilite[f"acces_{cat}_p95"], method="spearman"),
        "p99 vs p90": echantillon[f"acces_{cat}"].corr(sensibilite[f"acces_{cat}_p90"], method="spearman"),
    }
    for cat in categories
]).round(3)

acces_hyper_p95 : terminé
acces_super_p95 : terminé
acces_proximite_p95 : terminé
acces_specialise_p95 : terminé
acces_hyper_p90 : terminé
acces_super_p90 : terminé
acces_proximite_p90 : terminé
acces_specialise_p90 : terminé


,categorie,p99 vs p95,p99 vs p90
0,hyper,0.908,0.817
1,super,0.909,0.824
2,proximite,0.929,0.849
3,specialise,0.907,0.814


In [18]:
echantillon = dvf.sample(100_000, random_state=0)

sensibilite = calculer_indices(echantillon, calculer_rayons(0.95), "_p95").join(
    calculer_indices(echantillon, calculer_rayons(0.90), "_p90")
)

pd.DataFrame([
    {
        "categorie": cat,
        "p99 vs p95": echantillon[f"acces_{cat}"].corr(sensibilite[f"acces_{cat}_p95"], method="spearman"),
        "p99 vs p90": echantillon[f"acces_{cat}"].corr(sensibilite[f"acces_{cat}_p90"], method="spearman"),
    }
    for cat in categories
]).round(3)

acces_hyper_p95 : terminé
acces_super_p95 : terminé
acces_proximite_p95 : terminé
acces_specialise_p95 : terminé
acces_hyper_p90 : terminé
acces_super_p90 : terminé
acces_proximite_p90 : terminé
acces_specialise_p90 : terminé


,categorie,p99 vs p95,p99 vs p90
0,hyper,0.908,0.817
1,super,0.909,0.824
2,proximite,0.929,0.849
3,specialise,0.907,0.814


In [27]:
def indice_acces_composite(xy_ventes, portee_vente, w, lambda_com):
    """Indice gravitaire avec la portée la plus contraignante entre logement et commerce."""
    resultat = np.zeros(len(xy_ventes))
    for debut in range(0, len(xy_ventes), TAILLE_BLOC):
        bloc = xy_ventes[debut:debut + TAILLE_BLOC]
        paires = cKDTree(bloc).sparse_distance_matrix(
            arbre_commerces, max_distance=4 * portee_vente, output_type="ndarray"
        )
        portee_paire = np.minimum(portee_vente, lambda_com[paires["j"]])
        contributions = w[paires["j"]] * np.exp(-paires["v"] / portee_paire)
        resultat[debut:debut + len(bloc)] = np.bincount(
            paires["i"], weights=contributions, minlength=len(bloc)
        )
    return resultat

In [ ]:
import os
import subprocess

DOSSIER_LOCAL = "/home/onyxia/work/donnees"            # hors du dépôt Git
DOSSIER_BUCKET = "s3/acharki-ensae/Projet-analyse-BPE/processed"

def sauvegarder(df, nom_fichier, delai_max=300):
    """Sauvegarde df en Parquet en local, puis tente de l'envoyer dans le bucket."""
    # 1. Sauvegarde locale
    os.makedirs(DOSSIER_LOCAL, exist_ok=True)
    chemin_local = os.path.join(DOSSIER_LOCAL, nom_fichier)
    df.to_parquet(chemin_local)
    taille = os.path.getsize(chemin_local) / 1e6
    print(f"Sauvegarde locale OK : {chemin_local} ({taille:.1f} Mo)")

    # 2. Envoi vers le bucket, avec une limite de temps
    try:
        r = subprocess.run(
            ["mc", "cp", chemin_local, f"{DOSSIER_BUCKET}/{nom_fichier}"],
            capture_output=True, text=True, timeout=delai_max,
        )
        if r.returncode == 0:
            print(f"Envoi vers le bucket OK : {DOSSIER_BUCKET}/{nom_fichier}")
        else:
            print("Échec de l'envoi vers le bucket :", r.stderr.strip())
            
    except subprocess.TimeoutExpired:
        print(f"Le bucket n'a pas répondu en {delai_max} s : seule la copie locale existe.")

In [30]:
# Découper le rang d'accès aux supermarchés en 5 tranches, dans chaque niveau de densité
dvf["quintile_super"] = pd.cut(
    dvf["rang_super"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=["1 (moins bien desservis)", "2", "3", "4", "5 (mieux desservis)"],
)

# Prix médian au m² pour chaque tranche, dans chaque niveau de densité
(dvf.groupby(["DENS7", "quintile_super"], observed=True)["prix_m2"]
 .median()
 .unstack()
 .round(0))

quintile_super,1 (moins bien desservis),2,3,4,5 (mieux desservis)
DENS7,,,,,
1,3214.0,3149.0,3173.0,3647.0,7946.0
2,2920.0,2674.0,2428.0,2286.0,2131.0
3,2864.0,2682.0,2340.0,2267.0,2500.0
4,2990.0,2889.0,2944.0,2875.0,2625.0
5,2125.0,2073.0,2170.0,2211.0,2222.0
6,1567.0,1680.0,1767.0,1882.0,2147.0
7,1316.0,1333.0,1316.0,1357.0,1515.0


In [31]:
q5 = dvf[(dvf["DENS7"] == "1") & (dvf["quintile_super"] == "5 (mieux desservis)")]
print(f"Part de logements parisiens dans la dernière tranche : {(q5['code_departement'] == '75').mean():.1%}")

Part de logements parisiens dans la dernière tranche : 48.2%


In [32]:
niveau1 = dvf[dvf["DENS7"] == "1"]
print(f"Part de logements parisiens dans tout le niveau 1 : {(niveau1['code_departement'] == '75').mean():.1%}")

Part de logements parisiens dans tout le niveau 1 : 9.6%


In [34]:
import os
import subprocess

DOSSIER_LOCAL = "/home/onyxia/work/donnees"            # hors du dépôt Git
DOSSIER_BUCKET = "s3/acharki-ensae/Projet-analyse-BPE/processed"

def sauvegarder(df, nom_fichier, delai_max=300):
    """Sauvegarde df en Parquet en local, puis tente de l'envoyer dans le bucket."""
    # 1. Sauvegarde locale
    os.makedirs(DOSSIER_LOCAL, exist_ok=True)
    chemin_local = os.path.join(DOSSIER_LOCAL, nom_fichier)
    df.to_parquet(chemin_local)
    taille = os.path.getsize(chemin_local) / 1e6
    print(f"Sauvegarde locale OK : {chemin_local} ({taille:.1f} Mo)")

    # 2. Envoi vers le bucket, avec une limite de temps
    try:
        r = subprocess.run(
            ["mc", "cp", chemin_local, f"{DOSSIER_BUCKET}/{nom_fichier}"],
            capture_output=True, text=True, timeout=delai_max,
        )
        if r.returncode == 0:
            print(f"Envoi vers le bucket OK : {DOSSIER_BUCKET}/{nom_fichier}")
        else:
            print("Échec de l'envoi vers le bucket :", r.stderr.strip())
    except subprocess.TimeoutExpired:
        print(f"Le bucket n'a pas répondu en {delai_max} s : seule la copie locale existe.")

In [ ]:
sauvegarder(dvf, "dvf_enrichi_2025.parquet")

Sauvegarde locale OK : /home/onyxia/work/donnees/dvf_enrichi_2025.parquet (150.8 Mo)


In [2]:
import os
import subprocess
from getpass import getpass

depot = "/home/onyxia/work/projet__python_DS"

def git(*args):
    r = subprocess.run(["git", "-C", depot, *args], capture_output=True, text=True)
    return r.stdout + r.stderr

print(git("pull"))
print(git("add", "notebooks/03_croisement.ipynb", "src/stockage.py", "requirements.txt"))
print(git("status", "--short"))
print(git("commit", "-m", "Travail en cours : nouvelle version de l'indice d'accessibilité"))

token = os.environ.get("GIT_PERSONAL_ACCESS_TOKEN") or getpass("Colle ton token GitHub : ")
url = f"https://{token}@github.com/Samsamdu86/projet__python_DS.git"
print(git("push", url, "main").replace(token, "***"))

Updating a3c84d1..fe14f31
From https://github.com/Samsamdu86/projet__python_DS
   a3c84d1..fe14f31  main       -> origin/main
error: Your local changes to the following files would be overwritten by merge:
	notebooks/01_nettoyage_bpe.ipynb
	notebooks/02_nettoyage_dvf.ipynb
Please commit your changes or stash them before you merge.
Aborting

fatal: pathspec 'src/stockage.py' did not match any files

 M notebooks/01_nettoyage_bpe.ipynb
 M notebooks/02_nettoyage_dvf.ipynb
 M requirements.txt
?? notebooks/03_croisement.ipynb

On branch main
Your branch is behind 'origin/main' by 9 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_nettoyage_bpe.ipynb
	modified:   notebooks/02_nettoyage_dvf.ipynb
	modified:   requirements.txt

Untracked files:
  (use "git add <file>..." t

Colle ton token GitHub :  ········


To https://github.com/Samsamdu86/projet__python_DS.git
 ! [rejected]        main -> main (non-fast-forward)
error: failed to push some refs to 'https://github.com/Samsamdu86/projet__python_DS.git'
hint: Updates were rejected because the tip of your current branch is behind
hint: its remote counterpart. If you want to integrate the remote changes,
hint: use 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.

